# Entrega 06 — Erros, equidade, produto e riscos

**Projeto:** Previsão de demanda do RU/UFPB  
**Integrante:** Kaio Vitor  
**GitHub:** kaiovitormr  

## 1. Objetivo

Analisar os erros das previsões de demanda do Restaurante
Universitário, identificando padrões de subestimação e
superestimação por curso, tipo de refeição, período e
faixa de demanda.

Também serão avaliadas as diferenças de desempenho entre
grupos, as limitações dos dados sintéticos e os riscos
associados ao uso das previsões no planejamento do RU.

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

In [6]:
RAIZ = Path.cwd().resolve()

while not (RAIZ / "dados" / "bruto" / "mock").is_dir():
    if RAIZ == RAIZ.parent:
        raise FileNotFoundError("Raiz do projeto não encontrada.")
    RAIZ = RAIZ.parent

DADOS = RAIZ / "dados" / "processado"

print("Raiz do projeto:", RAIZ)
print("\nArquivos disponíveis:")

for arquivo in sorted(DADOS.glob("*.csv")):
    print(arquivo.name)

Raiz do projeto: C:\dev\previsao-demanda-ru-ufpb

Arquivos disponíveis:
base_analitica_diaria.csv
base_analitica_diaria_curso.csv
resultados_baselines_total_diario.csv
resultados_baselines_validacao_por_curso.csv


## 2. Metodologia da análise

A análise considera a quantidade de refeições servidas
por data, tipo de refeição e curso.

Serão utilizadas as previsões produzidas pelos modelos
selecionados pela equipe, respeitando a partição temporal
e o protocolo de avaliação compartilhado.

Os erros serão avaliados por meio do MAE, RMSE e erro
médio com sinal, considerando também a quantidade de
observações em cada grupo.

Como os dados são sintéticos, os resultados não permitem
concluir que o modelo apresenta equidade ou desempenho
equivalente em condições reais de funcionamento do RU.

## 3. Cálculo dos erros das previsões

O erro é calculado pela diferença entre a quantidade
prevista e a quantidade observada de refeições.

Valores negativos indicam subestimação, enquanto
valores positivos indicam superestimação.

A análise permite identificar possíveis impactos
no planejamento da produção de refeições do RU.

In [7]:
def calcular_erros(previsoes):
    df = previsoes.copy()

    df["erro"] = df["previsto"] - df["observado"]

    df["erro_absoluto"] = df["erro"].abs()

    df["erro_quadratico"] = df["erro"] ** 2

    df["tipo_erro"] = np.select(
        [
            df["erro"] < 0,
            df["erro"] > 0
        ],
        [
            "Subestimacao",
            "Superestimacao"
        ],
        default="Acerto exato"
    )

    return df

## 4. Métricas de avaliação

- MAE: magnitude média dos erros em refeições.
- RMSE: penaliza erros de maior magnitude.
- Viés: indica a tendência de subestimar ou
  superestimar a quantidade de refeições.
- N: quantidade de observações avaliadas.

A quantidade de observações é importante para
interpretar as diferenças entre os grupos.

In [8]:
def resumir_erros(df, grupos):
    resumo = (
        df.groupby(grupos, observed=True)
        .agg(
            n=("erro", "size"),
            MAE=("erro_absoluto", "mean"),
            MSE=("erro_quadratico", "mean"),
            vies=("erro", "mean"),
            subestimacoes=("erro", lambda x: (x < 0).sum()),
            superestimacoes=("erro", lambda x: (x > 0).sum())
        )
        .reset_index()
    )

    resumo["RMSE"] = np.sqrt(resumo["MSE"])

    return resumo.drop(columns="MSE")

In [9]:
teste = pd.DataFrame({
    "modelo": ["Teste", "Teste", "Teste"],
    "observado": [100, 150, 200],
    "previsto": [90, 160, 200]
})

teste_erros = calcular_erros(teste)

display(teste_erros)
display(resumir_erros(teste_erros, ["modelo"]))

,modelo,observado,previsto,erro,erro_absoluto,erro_quadratico,tipo_erro
0,Teste,100,90,-10,10,100,Subestimacao
1,Teste,150,160,10,10,100,Superestimacao
2,Teste,200,200,0,0,0,Acerto exato


,modelo,n,MAE,vies,subestimacoes,superestimacoes,RMSE
0,Teste,3,6.67,0.00,1,1,8.16


In [10]:
resumo_teste = resumir_erros(teste_erros, ["modelo"])

assert np.isclose(resumo_teste["MAE"].iloc[0], 20/3)
assert np.isclose(resumo_teste["RMSE"].iloc[0], np.sqrt(200/3))
assert np.isclose(resumo_teste["vies"].iloc[0], 0)
assert resumo_teste["n"].iloc[0] == 3

display(resumo_teste)

,modelo,n,MAE,vies,subestimacoes,superestimacoes,RMSE
0,Teste,3,6.67,0.00,1,1,8.16


## 5. Análise de erros por grupos

Para identificar diferenças no desempenho dos modelos,
os erros serão analisados considerando:

- Tipo de refeição;
- Curso;
- Período;
- Faixa de demanda.

Cada grupo será acompanhado pela quantidade de
observações, evitando conclusões precipitadas sobre
grupos com poucas ocorrências.

As partições temporais serão analisadas separadamente.

In [11]:
def analisar_grupos(df):
    resultados = {}

    dimensoes = [
        "tipo_refeicao",
        "curso",
        "periodo",
        "faixa_demanda"
    ]

    for dimensao in dimensoes:
        resultados[dimensao] = resumir_erros(
            df,
            ["modelo", "particao", dimensao]
        ).sort_values(
            "MAE", ascending=False
        )

    return resultados

## 6. Investigação dos maiores erros

A identificação das maiores diferenças entre os
valores observados e previstos permite investigar
situações críticas para o planejamento do RU.

A subestimação pode contribuir para a falta de
refeições, enquanto a superestimação pode resultar
em produção excedente e desperdício.

Esses impactos são potenciais, pois o mock não
representa diretamente a operação real do RU.

In [13]:
def identificar_casos_criticos(df, n=5):
    resultados = {}

    for chave, grupo in df.groupby(
        ["modelo", "particao"]
    ):
        resultados[chave] = {
            "maiores_erros": grupo.nlargest(
                n, "erro_absoluto"
            ),
            "subestimacoes": grupo[
                grupo["erro"] < 0
            ].nsmallest(n, "erro"),
            "superestimacoes": grupo[
                grupo["erro"] > 0
            ].nlargest(n, "erro")
        }

    return resultados

## 7. Preparação das previsões para análise

As previsões devem seguir o protocolo compartilhado
pela equipe, contendo data, tipo de refeição, curso,
partição, modelo, quantidade observada e prevista.

Para a análise temporal, será utilizado o mês de
referência. As faixas de demanda serão definidas
com base em limites calculados no conjunto de treino,
evitando utilizar informações futuras.

In [14]:
def preparar_previsoes(previsoes, limite_baixo, limite_alto):
    df = previsoes.copy()

    obrigatorias = [
        "data",
        "tipo_refeicao",
        "curso",
        "particao",
        "modelo",
        "observado",
        "previsto"
    ]

    faltantes = [
        c for c in obrigatorias
        if c not in df.columns
    ]

    if faltantes:
        raise ValueError(f"Colunas ausentes: {faltantes}")

    if df[obrigatorias].isna().any().any():
        raise ValueError("Existem valores ausentes.")

    df["data"] = pd.to_datetime(df["data"])

    for coluna in ["observado", "previsto"]:
        df[coluna] = pd.to_numeric(df[coluna])

        if not np.isfinite(df[coluna]).all():
            raise ValueError(f"Valores invalidos: {coluna}")

    df["periodo"] = df["data"].dt.strftime("%Y-%m")

    df["faixa_demanda"] = np.select(
        [
            df["observado"] <= limite_baixo,
            df["observado"] <= limite_alto
        ],
        [
            "Baixa",
            "Media"
        ],
        default="Alta"
    )

    return calcular_erros(df)

## 8. Equidade e comparação entre grupos

A comparação entre cursos e tipos de refeição busca
identificar diferenças no desempenho dos modelos.

Serão considerados o MAE, RMSE, viés e quantidade
de observações de cada grupo.

Variáveis como curso, turno e residência estudantil,
quando utilizadas pelos modelos, podem funcionar como
proxies de características socioeconômicas.

Entretanto, por se tratar de dados sintéticos, não é
possível afirmar a existência ou ausência de
discriminação na operação real do RU.

Grupos com poucas observações devem ser interpretados
com cautela.

In [15]:
def comparar_equidade(df, minimo=30):
    resumo = resumir_erros(
        df,
        ["modelo", "particao", "curso"]
    )

    resumo["amostra_suficiente"] = (
        resumo["n"] >= minimo
    )

    return resumo.sort_values(
        ["modelo", "particao", "MAE"],
        ascending=[True, True, False]
    )

In [16]:
teste_integrado = pd.DataFrame({
    "data": ["2026-10-01", "2026-10-02",
             "2026-10-03", "2026-10-04"],
    "tipo_refeicao": ["Almoco", "Jantar",
                     "Almoco", "Jantar"],
    "curso": ["Curso A", "Curso B",
              "Curso A", "Curso B"],
    "particao": ["validacao"] * 4,
    "modelo": ["Teste"] * 4,
    "observado": [100, 150, 200, 250],
    "previsto": [90, 170, 180, 280]
})

teste_preparado = preparar_previsoes(
    teste_integrado,
    limite_baixo=120,
    limite_alto=200
)

resultados_teste = analisar_grupos(teste_preparado)

for dimensao, resultado in resultados_teste.items():
    print(f"\nAnalise por {dimensao}:")
    display(resultado)

display(comparar_equidade(teste_preparado))

casos_teste = identificar_casos_criticos(teste_preparado)

for chave, casos in casos_teste.items():
    print(chave)
    display(casos["maiores_erros"])


Analise por tipo_refeicao:


,modelo,particao,tipo_refeicao,n,MAE,vies,subestimacoes,superestimacoes,RMSE
1,Teste,validacao,Jantar,2,25.00,25.00,0,2,25.50
0,Teste,validacao,Almoco,2,15.00,-15.00,2,0,15.81



Analise por curso:


,modelo,particao,curso,n,MAE,vies,subestimacoes,superestimacoes,RMSE
1,Teste,validacao,Curso B,2,25.00,25.00,0,2,25.50
0,Teste,validacao,Curso A,2,15.00,-15.00,2,0,15.81



Analise por periodo:


,modelo,particao,periodo,n,MAE,vies,subestimacoes,superestimacoes,RMSE
0,Teste,validacao,2026-10,4,20.00,5.00,2,2,21.21



Analise por faixa_demanda:


,modelo,particao,faixa_demanda,n,MAE,vies,subestimacoes,superestimacoes,RMSE
0,Teste,validacao,Alta,1,30.00,30.00,0,1,30.00
2,Teste,validacao,Media,2,20.00,0.00,1,1,20.00
1,Teste,validacao,Baixa,1,10.00,-10.00,1,0,10.00


,modelo,particao,curso,n,MAE,vies,subestimacoes,superestimacoes,RMSE,amostra_suficiente
1,Teste,validacao,Curso B,2,25.00,25.00,0,2,25.50,False
0,Teste,validacao,Curso A,2,15.00,-15.00,2,0,15.81,False


('Teste', 'validacao')


,data,tipo_refeicao,curso,particao,modelo,observado,previsto,periodo,faixa_demanda,erro,erro_absoluto,erro_quadratico,tipo_erro
3,2026-10-04,Jantar,Curso B,validacao,Teste,250,280,2026-10,Alta,30,30,900,Superestimacao
1,2026-10-02,Jantar,Curso B,validacao,Teste,150,170,2026-10,Media,20,20,400,Superestimacao
2,2026-10-03,Almoco,Curso A,validacao,Teste,200,180,2026-10,Media,-20,20,400,Subestimacao
0,2026-10-01,Almoco,Curso A,validacao,Teste,100,90,2026-10,Baixa,-10,10,100,Subestimacao


## 9. Proposta de produto

O produto proposto consiste em um painel de previsão
de demanda do Restaurante Universitário da UFPB,
voltado à equipe responsável pelo planejamento e
pela produção das refeições.

A ferramenta deverá auxiliar na estimativa da
quantidade de refeições necessárias para o dia
seguinte, contribuindo para o planejamento dos
recursos disponíveis.

### Entradas

- Data de referência;
- Tipo de refeição;
- Curso;
- Informações históricas disponíveis até o dia anterior.

### Saídas

- Quantidade prevista de refeições;
- Histórico da demanda;
- Comparação entre valores observados e previstos;
- Indicadores de erro;
- Alertas sobre limitações e cobertura dos dados.

A previsão deverá funcionar como apoio à decisão,
sem substituir a avaliação dos responsáveis pelo RU.

### 9.1 Wireframe conceitual

```
+------------------------------------------------+
| RU/UFPB - PAINEL DE PREVISAO DE DEMANDA         |
+------------------------------------------------+
| Data: [      ] Refeicao: [      ] Curso: [    ] |
+------------------------------------------------+
| Demanda prevista | MAE historico | Vies        |
|        --        |       --      |  --         |
+------------------------------------------------+
|                                                |
|       GRAFICO DE HISTORICO E PREVISAO           |
|                                                |
+------------------------------------------------+
| Alertas de cobertura e limitacoes dos dados    |
+------------------------------------------------+
```

O wireframe representa uma proposta de interface,
sem implementação funcional nesta etapa.

## 10. Riscos e limitações

| Risco | Impacto | Mitigação |
|---|---|---|
| Dados sintéticos | Resultados diferentes da realidade | Validar com dados reais |
| Cobertura incompleta | Previsões distorcidas | Verificar a qualidade das extrações |
| Degradação do modelo | Perda de precisão ao longo do tempo | Monitorar os indicadores de erro |
| Subestimação | Possível insuficiência de refeições | Revisar previsões críticas |
| Superestimação | Possível desperdício de alimentos | Acompanhar os excedentes |
| Uso inadequado | Decisões baseadas apenas no modelo | Manter supervisão humana |
| Diferenças entre grupos | Desempenho desigual das previsões | Monitorar erros por curso e refeição |

O modelo não deve ser utilizado para determinar
o direito de acesso dos estudantes ao RU.

Os resultados obtidos com dados sintéticos não
comprovam equidade, eficiência operacional ou
redução efetiva de desperdícios em condições reais.

## 11. Execução da análise dos modelos

Nesta etapa, serão utilizadas as previsões dos
modelos selecionados pela equipe com base no
desempenho na validação temporal.

A análise será realizada por curso, tipo de
refeição, período e faixa de demanda.

Os resultados serão utilizados para identificar
diferenças de desempenho, tendências de erro
e possíveis limitações dos modelos.

A execução definitiva depende da disponibilização
das previsões e dos limites de demanda definidos
a partir dos dados de treino.

In [17]:
def executar_analise(previsoes, limite_baixo, limite_alto):

    df = preparar_previsoes(
        previsoes,
        limite_baixo,
        limite_alto
    )

    print("RESUMO GERAL")
    display(resumir_erros(df, ["modelo", "particao"]))

    print("\nANALISE POR GRUPOS")

    resultados = analisar_grupos(df)

    for dimensao, tabela in resultados.items():
        print(f"\nAnalise por {dimensao}")
        display(tabela)

    print("\nANALISE DE EQUIDADE")
    display(comparar_equidade(df))

    print("\nCASOS CRITICOS")

    casos = identificar_casos_criticos(df)

    for chave, tabelas in casos.items():
        print(chave)

        for nome, tabela in tabelas.items():
            print(nome)
            display(tabela)

    return df

In [18]:
def visualizar_erros(df):

    for chave, grupo in df.groupby(
        ["modelo", "particao"]
    ):

        resumo = resumir_erros(grupo, ["curso"])

        print(f"Modelo e particao: {chave}")

        display(
            resumo.style.bar(
                subset=["MAE", "RMSE"],
                color="#93c5fd"
            )
        )

In [20]:
arquivos = [
    "base_analitica_diaria_curso.csv",
    "resultados_baselines_validacao_por_curso.csv"
]

for arquivo in arquivos:
    base = pd.read_csv(DADOS / arquivo)

    display(base.head())

,data,tipo_refeicao,curso,aberto,extracao_completa,quantidade_servida,valor_observado
0,2019-01-01,almoco,CURSO_TECNICO_A,True,True,0,NaN
1,2019-01-01,almoco,CURSO_TECNICO_B,True,True,6,NaN
2,2019-01-01,almoco,CURSO_TECNICO_C,True,True,1,NaN
3,2019-01-01,almoco,CURSO_TECNICO_D,True,True,3,NaN
4,2019-01-01,almoco,CURSO_TECNICO_E,True,True,0,NaN


,baseline,curso,refeicao,linhas,MAE,RMSE
0,Média do treino,CURSO_TECNICO_A,almoco,313,3.96,4.42
1,Média do treino,CURSO_TECNICO_A,cafe_da_manha,313,2.09,2.45
2,Média do treino,CURSO_TECNICO_A,jantar,313,1.77,2.07
3,Média do treino,CURSO_TECNICO_B,almoco,313,4.11,4.55
4,Média do treino,CURSO_TECNICO_B,cafe_da_manha,313,2.14,2.61


In [21]:
base_curso = pd.read_csv(
    DADOS / "base_analitica_diaria_curso.csv"
)

print("Total de registros:", len(base_curso))

print("\nValores ausentes:")
print(base_curso.isna().sum())

print("\nTipos de refeicao:")
print(base_curso["tipo_refeicao"].value_counts())

print("\nCobertura:")
print(base_curso["extracao_completa"].value_counts())

print("\nFuncionamento:")
print(base_curso["aberto"].value_counts())

print("\nEstatisticas da demanda:")
display(base_curso["quantidade_servida"].describe())

Total de registros: 51750

Valores ausentes:
data                      0
tipo_refeicao             0
curso                     0
aberto                    0
extracao_completa         0
quantidade_servida        0
valor_observado       51750
dtype: int64

Tipos de refeicao:
tipo_refeicao
almoco           17250
cafe_da_manha    17250
jantar           17250
Name: count, dtype: int64

Cobertura:
extracao_completa
True    51750
Name: count, dtype: int64

Funcionamento:
aberto
True     44370
False     7380
Name: count, dtype: int64

Estatisticas da demanda:


count   51750.00
mean        4.28
std         3.88
min         0.00
25%         1.00
50%         4.00
75%         7.00
max        19.00
Name: quantidade_servida, dtype: float64

In [22]:
baseline = pd.read_csv(
    DADOS / "resultados_baselines_validacao_por_curso.csv"
)

print("Total de registros:", len(baseline))

print("\nColunas:")
print(baseline.columns.tolist())

display(baseline.head(10))

Total de registros: 38

Colunas:
['baseline', 'curso', 'refeicao', 'linhas', 'MAE', 'RMSE']


,baseline,curso,refeicao,linhas,MAE,RMSE
0,Média do treino,CURSO_TECNICO_A,almoco,313,3.96,4.42
1,Média do treino,CURSO_TECNICO_A,cafe_da_manha,313,2.09,2.45
2,Média do treino,CURSO_TECNICO_A,jantar,313,1.77,2.07
3,Média do treino,CURSO_TECNICO_B,almoco,313,4.11,4.55
4,Média do treino,CURSO_TECNICO_B,cafe_da_manha,313,2.14,2.61
5,Média do treino,CURSO_TECNICO_B,jantar,313,2.10,2.48
6,Média do treino,CURSO_TECNICO_C,almoco,313,3.96,4.42
7,Média do treino,CURSO_TECNICO_C,cafe_da_manha,313,2.16,2.59
8,Média do treino,CURSO_TECNICO_C,jantar,313,1.80,2.15
9,Média do treino,CURSO_TECNICO_D,almoco,313,4.01,4.51


## 13. Diagnóstico da base mock

A base analítica apresenta 51.750 registros,
distribuídos igualmente entre café da manhã,
almoço e jantar.

Todos os registros apresentam cobertura completa
no cenário simulado. Do total, 44.370 correspondem
a períodos de funcionamento e 7.380 a períodos
em que o RU está fechado.

A variável quantidade_servida não apresenta
valores ausentes.

A coluna valor_observado apresenta ausência
em todos os registros e não será utilizada
como alvo da análise.

In [23]:
base_aberta = base_curso[
    (base_curso["aberto"] == True) &
    (base_curso["extracao_completa"] == True)
].copy()

demanda_refeicao = (
    base_aberta.groupby("tipo_refeicao")
    .agg(
        registros=("quantidade_servida", "size"),
        total_refeicoes=("quantidade_servida", "sum"),
        demanda_media=("quantidade_servida", "mean"),
        demanda_maxima=("quantidade_servida", "max"),
        desvio_padrao=("quantidade_servida", "std")
    )
    .reset_index()
)

display(demanda_refeicao)

,tipo_refeicao,registros,total_refeicoes,demanda_media,demanda_maxima,desvio_padrao
0,almoco,14790,110161,7.45,19,4.42
1,cafe_da_manha,14790,58707,3.97,14,2.68
2,jantar,14790,52732,3.57,14,2.46


In [24]:
demanda_curso = (
    base_aberta.groupby("curso")
    .agg(
        registros=("quantidade_servida", "size"),
        total_refeicoes=("quantidade_servida", "sum"),
        demanda_media=("quantidade_servida", "mean"),
        desvio_padrao=("quantidade_servida", "std")
    )
    .reset_index()
    .sort_values("demanda_media", ascending=False)
)

display(demanda_curso)

,curso,registros,total_refeicoes,demanda_media,desvio_padrao
1,CURSO_TECNICO_B,7395,45480,6.15,3.63
5,CURSO_TECNICO_F,7395,45442,6.14,3.60
3,CURSO_TECNICO_D,7395,45426,6.14,3.65
0,CURSO_TECNICO_A,7395,28629,3.87,3.50
2,CURSO_TECNICO_C,7395,28454,3.85,3.53
4,CURSO_TECNICO_E,7395,28169,3.81,3.44


In [25]:
print("Baselines disponiveis:")
print(baseline["baseline"].unique())

print("\nCursos disponiveis:")
print(baseline["curso"].unique())

print("\nRefeicoes disponiveis:")
print(baseline["refeicao"].unique())

display(
    baseline.sort_values("MAE", ascending=False)
)

Baselines disponiveis:
<StringArray>
['Média do treino', 'D-7 proposto']
Length: 2, dtype: str

Cursos disponiveis:
<StringArray>
['CURSO_TECNICO_A', 'CURSO_TECNICO_B', 'CURSO_TECNICO_C', 'CURSO_TECNICO_D',
 'CURSO_TECNICO_E', 'CURSO_TECNICO_F',           'TODOS']
Length: 7, dtype: str

Refeicoes disponiveis:
<StringArray>
['almoco', 'cafe_da_manha', 'jantar', 'TODAS']
Length: 4, dtype: str


,baseline,curso,refeicao,linhas,MAE,RMSE
3,Média do treino,CURSO_TECNICO_B,almoco,313,4.11,4.55
9,Média do treino,CURSO_TECNICO_D,almoco,313,4.01,4.51
6,Média do treino,CURSO_TECNICO_C,almoco,313,3.96,4.42
0,Média do treino,CURSO_TECNICO_A,almoco,313,3.96,4.42
12,Média do treino,CURSO_TECNICO_E,almoco,313,3.95,4.42
15,Média do treino,CURSO_TECNICO_F,almoco,313,3.88,4.36
28,D-7 proposto,CURSO_TECNICO_D,almoco,313,2.80,3.80
18,Média do treino,TODOS,TODAS,5634,2.67,3.24
22,D-7 proposto,CURSO_TECNICO_B,almoco,313,2.60,3.51
19,D-7 proposto,CURSO_TECNICO_A,almoco,313,2.54,3.57


In [26]:
chaves = ["data", "tipo_refeicao", "curso"]

print("Registros duplicados:",
      base_curso.duplicated(chaves).sum())

print("Demandas negativas:",
      (base_curso["quantidade_servida"] < 0).sum())

print("Refeicoes servidas com RU fechado:",
      len(base_curso[
          (base_curso["aberto"] == False) &
          (base_curso["quantidade_servida"] > 0)
      ]))

Registros duplicados: 0
Demandas negativas: 0
Refeicoes servidas com RU fechado: 0


## 14. Análise exploratória da demanda

A base mock contabiliza 221.600 refeições servidas,
sendo 110.161 almoços, 58.707 cafés da manhã
e 52.732 jantares.

O almoço concentra aproximadamente 49,71% das
refeições e apresenta maior variabilidade absoluta,
com desvio padrão de 4,42 refeições por observação.

Os cursos B, F e D apresentam demanda média
superior à dos cursos A, C e E.

Essas diferenças justificam a avaliação dos erros
por grupo, considerando tanto a magnitude dos erros
quanto o perfil de demanda de cada curso.

Não foram identificados registros duplicados,
demandas negativas ou refeições servidas durante
períodos de fechamento do RU.

Essas características são referentes exclusivamente
ao cenário simulado.

In [31]:
# Analise preliminar consolidada dos erros

baseline = pd.read_csv(
    DADOS / "resultados_baselines_validacao_por_curso.csv"
)

erros_detalhados = baseline[
    (baseline["curso"] != "TODOS") &
    (baseline["refeicao"] != "TODAS")
].copy()

print("=" * 60)
print("ANALISE PRELIMINAR DE ERROS E EQUIDADE")
print("=" * 60)

for modelo, grupo in erros_detalhados.groupby("baseline"):

    print(f"\nBASELINE: {modelo}")
    print("-" * 60)

    # Comparacao entre cursos
    print("\n1. ERROS POR CURSO")

    por_curso = baseline[
        (baseline["baseline"] == modelo) &
        (baseline["curso"] != "TODOS") &
        (baseline["refeicao"] == "TODAS")
    ]

    if not por_curso.empty:
        display(por_curso.sort_values("MAE", ascending=False))
    else:
        print("Sem metricas consolidadas por curso.")

    # Comparacao entre refeicoes
    print("\n2. ERROS POR REFEICAO")

    por_refeicao = baseline[
        (baseline["baseline"] == modelo) &
        (baseline["curso"] == "TODOS") &
        (baseline["refeicao"] != "TODAS")
    ]

    if not por_refeicao.empty:
        display(por_refeicao.sort_values("MAE", ascending=False))
    else:
        print("Sem metricas consolidadas por refeicao.")

    # Maiores erros
    print("\n3. GRUPOS COM MAIORES ERROS")
    display(grupo.nlargest(3, "MAE"))

    # Menores erros
    print("\n4. GRUPOS COM MENORES ERROS")
    display(grupo.nsmallest(3, "MAE"))

    # Diferencas de desempenho
    maior = grupo.loc[grupo["MAE"].idxmax()]
    menor = grupo.loc[grupo["MAE"].idxmin()]

    diferenca = maior["MAE"] - menor["MAE"]

    print("\n5. INTERPRETACAO DOS RESULTADOS")

    print(
        f"Maior erro: {maior['curso']} - "
        f"{maior['refeicao']}, "
        f"MAE de {maior['MAE']:.2f} refeicoes, "
        f"com {maior['linhas']} observacoes."
    )

    print(
        f"Menor erro: {menor['curso']} - "
        f"{menor['refeicao']}, "
        f"MAE de {menor['MAE']:.2f} refeicoes, "
        f"com {menor['linhas']} observacoes."
    )

    print(
        f"Diferenca entre os grupos: "
        f"{diferenca:.2f} refeicoes."
    )

    print(f"Grupos avaliados: {len(grupo)}")

    if menor["MAE"] > 0:
        percentual = (
            diferenca / menor["MAE"]
        ) * 100

        print(
            f"O maior MAE supera o menor "
            f"em {percentual:.2f}%."
        )

# Resumo consolidado

print("\n" + "=" * 60)
print("6. RESUMO CONSOLIDADO")
print("=" * 60)

comparacao = (
    erros_detalhados.groupby("baseline")
    .agg(
        menor_MAE=("MAE", "min"),
        maior_MAE=("MAE", "max"),
        grupos_avaliados=("MAE", "size")
    )
    .reset_index()
)

comparacao["diferenca_MAE"] = (
    comparacao["maior_MAE"] -
    comparacao["menor_MAE"]
)

display(comparacao)

print("\n7. LIMITACOES DA ANALISE")

print("""
- Resultados preliminares dos baselines anteriores.
- Dados exclusivamente sinteticos.
- Diferencas de erro nao comprovam discriminacao.
- O arquivo nao permite calcular vies individual.
- A analise definitiva depende das previsoes da E6.
""")

ANALISE PRELIMINAR DE ERROS E EQUIDADE

BASELINE: D-7 proposto
------------------------------------------------------------

1. ERROS POR CURSO
Sem metricas consolidadas por curso.

2. ERROS POR REFEICAO
Sem metricas consolidadas por refeicao.

3. GRUPOS COM MAIORES ERROS


,baseline,curso,refeicao,linhas,MAE,RMSE
28,D-7 proposto,CURSO_TECNICO_D,almoco,313,2.80,3.80
22,D-7 proposto,CURSO_TECNICO_B,almoco,313,2.60,3.51
19,D-7 proposto,CURSO_TECNICO_A,almoco,313,2.54,3.57



4. GRUPOS COM MENORES ERROS


,baseline,curso,refeicao,linhas,MAE,RMSE
21,D-7 proposto,CURSO_TECNICO_A,jantar,313,1.52,2.11
33,D-7 proposto,CURSO_TECNICO_E,jantar,313,1.62,2.16
27,D-7 proposto,CURSO_TECNICO_C,jantar,313,1.65,2.26



5. INTERPRETACAO DOS RESULTADOS
Maior erro: CURSO_TECNICO_D - almoco, MAE de 2.80 refeicoes, com 313 observacoes.
Menor erro: CURSO_TECNICO_A - jantar, MAE de 1.52 refeicoes, com 313 observacoes.
Diferenca entre os grupos: 1.28 refeicoes.
Grupos avaliados: 18
O maior MAE supera o menor em 84.21%.

BASELINE: Média do treino
------------------------------------------------------------

1. ERROS POR CURSO
Sem metricas consolidadas por curso.

2. ERROS POR REFEICAO
Sem metricas consolidadas por refeicao.

3. GRUPOS COM MAIORES ERROS


,baseline,curso,refeicao,linhas,MAE,RMSE
3,Média do treino,CURSO_TECNICO_B,almoco,313,4.11,4.55
9,Média do treino,CURSO_TECNICO_D,almoco,313,4.01,4.51
0,Média do treino,CURSO_TECNICO_A,almoco,313,3.96,4.42



4. GRUPOS COM MENORES ERROS


,baseline,curso,refeicao,linhas,MAE,RMSE
14,Média do treino,CURSO_TECNICO_E,jantar,313,1.73,2.06
2,Média do treino,CURSO_TECNICO_A,jantar,313,1.77,2.07
8,Média do treino,CURSO_TECNICO_C,jantar,313,1.80,2.15



5. INTERPRETACAO DOS RESULTADOS
Maior erro: CURSO_TECNICO_B - almoco, MAE de 4.11 refeicoes, com 313 observacoes.
Menor erro: CURSO_TECNICO_E - jantar, MAE de 1.73 refeicoes, com 313 observacoes.
Diferenca entre os grupos: 2.38 refeicoes.
Grupos avaliados: 18
O maior MAE supera o menor em 137.57%.

6. RESUMO CONSOLIDADO


,baseline,menor_MAE,maior_MAE,grupos_avaliados,diferenca_MAE
0,D-7 proposto,1.52,2.80,18,1.28
1,Média do treino,1.73,4.11,18,2.38



7. LIMITACOES DA ANALISE

- Resultados preliminares dos baselines anteriores.
- Dados exclusivamente sinteticos.
- Diferencas de erro nao comprovam discriminacao.
- O arquivo nao permite calcular vies individual.
- A analise definitiva depende das previsoes da E6.



## 15. Interpretação dos resultados preliminares

A análise dos baselines anteriores considerou 18
grupos por método, correspondentes às combinações
entre seis cursos e três tipos de refeição.

Cada grupo apresentado nos extremos de desempenho
possui 313 observações.

No baseline D-7 proposto, o maior MAE foi observado
no almoço do curso D, com 2,80 refeições. O menor
ocorreu no jantar do curso A, com 1,52 refeições.

A diferença entre esses grupos foi de 1,28
refeições, equivalente a aproximadamente 84,21%.

Na média do treino, o maior MAE foi de 4,11
refeições, no almoço do curso B, enquanto o menor
foi de 1,73, no jantar do curso E.

A diferença entre os extremos foi de 2,38
refeições, equivalente a aproximadamente 137,57%.

O D-7 apresentou menor amplitude de erros entre
os grupos avaliados. Entretanto, esse resultado
isolado não comprova equidade nem determina
a escolha do melhor modelo.

Nos dois baselines, os maiores erros estão
associados ao almoço, que também apresenta
maior demanda e variabilidade na base mock.

Essa característica indica a necessidade de
atenção especial às previsões dessa refeição
durante o planejamento da produção.

Os resultados são preliminares e não representam
a avaliação definitiva dos modelos da Entrega 06.

In [32]:
def consolidar_metricas(df, dimensao):

    dados = df.copy()

    dados["erro_absoluto_total"] = (
        dados["MAE"] * dados["linhas"]
    )

    dados["erro_quadratico_total"] = (
        dados["RMSE"] ** 2 * dados["linhas"]
    )

    resumo = (
        dados.groupby(["baseline", dimensao])
        .agg(
            n=("linhas", "sum"),
            soma_erros=("erro_absoluto_total", "sum"),
            soma_quadrados=("erro_quadratico_total", "sum")
        )
        .reset_index()
    )

    resumo["MAE"] = resumo["soma_erros"] / resumo["n"]

    resumo["RMSE"] = np.sqrt(
        resumo["soma_quadrados"] / resumo["n"]
    )

    return resumo.drop(
        columns=["soma_erros", "soma_quadrados"]
    ).sort_values(
        ["baseline", "MAE"],
        ascending=[True, False]
    )


print("ERROS CONSOLIDADOS POR CURSO")

display(
    consolidar_metricas(erros_detalhados, "curso")
)

print("\nERROS CONSOLIDADOS POR REFEICAO")

display(
    consolidar_metricas(erros_detalhados, "refeicao")
)

ERROS CONSOLIDADOS POR CURSO


,baseline,curso,n,MAE,RMSE
3,D-7 proposto,CURSO_TECNICO_D,939,2.35,3.15
1,D-7 proposto,CURSO_TECNICO_B,939,2.34,3.09
5,D-7 proposto,CURSO_TECNICO_F,939,2.30,3.11
2,D-7 proposto,CURSO_TECNICO_C,939,1.96,2.80
0,D-7 proposto,CURSO_TECNICO_A,939,1.94,2.79
4,D-7 proposto,CURSO_TECNICO_E,939,1.94,2.73
7,Média do treino,CURSO_TECNICO_B,939,2.78,3.35
9,Média do treino,CURSO_TECNICO_D,939,2.75,3.34
11,Média do treino,CURSO_TECNICO_F,939,2.70,3.28
8,Média do treino,CURSO_TECNICO_C,939,2.64,3.21



ERROS CONSOLIDADOS POR REFEICAO


,baseline,refeicao,n,MAE,RMSE
0,D-7 proposto,almoco,1878,2.56,3.55
1,D-7 proposto,cafe_da_manha,1878,2.01,2.68
2,D-7 proposto,jantar,1878,1.85,2.51
3,Média do treino,almoco,1878,3.98,4.45
4,Média do treino,cafe_da_manha,1878,2.13,2.56
5,Média do treino,jantar,1878,1.89,2.27


In [33]:
funcoes = [
    "calcular_erros",
    "resumir_erros",
    "analisar_grupos",
    "identificar_casos_criticos",
    "preparar_previsoes",
    "comparar_equidade",
    "executar_analise",
    "visualizar_erros",
    "consolidar_metricas"
]

for nome in funcoes:
    status = "OK" if callable(globals().get(nome)) else "PENDENTE"
    print(f"{nome}: {status}")

calcular_erros: OK
resumir_erros: OK
analisar_grupos: OK
identificar_casos_criticos: OK
preparar_previsoes: OK
comparar_equidade: OK
executar_analise: OK
visualizar_erros: OK
consolidar_metricas: OK


## 16. Conclusão da contribuição individual

Foram desenvolvidas funções para análise de erros,
comparação de desempenho entre grupos e identificação
de casos críticos de previsão.

A análise exploratória da base mock e dos baselines
anteriores identificou diferenças de desempenho
entre cursos e tipos de refeição.

Também foram elaboradas a proposta de produto,
o wireframe e a discussão dos riscos e limitações.

A avaliação definitiva permanece condicionada
à disponibilização das previsões oficiais da E6.

Nenhum modelo foi treinado ou selecionado nesta
contribuição individual.